# Analysis
It is time to analyse some data. We here show how to set up an Analysis object and use it to first fit an artificial vanadium measurement. Next, we use the fitted resolution to fit an artificial measurement of a model with diffusion and some elastic scattering. 

We extract and plot the relevant parameters. Finally, we show how to fit directly to the diffusion model.

In the near future, it will be possible to fit the width and area of the Lorentzian to the diffusion model as well.

In [1]:
# Imports
import pooch

import easydynamics as edyn

%matplotlib widget

In [2]:
# Load the vanadium data
vanadium_experiment = edyn.Experiment('Vanadium')
file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/vanadium_data_example.h5',
    known_hash='16cc1b327c303feeb88fb9dda5390dc4880b62396b1793f98c6fef0b27c7b873',
)

vanadium_experiment.load_hdf5(filename=file_path)

In [3]:
# Example of Analysis with a simple sample model and instrument model
# The scattering from vanadium is purely elastic, so we model it with a
# delta function
delta_function = edyn.DeltaFunction(display_name='DeltaFunction', area=1)
sample_model = edyn.SampleModel(
    components=delta_function,
)

# The resolution is in this case modeled as a Gaussian. However, we can
# add as many components as we like to the resolution model
res_gauss = edyn.Gaussian(width=0.1)
res_gauss.area.fixed = True
resolution_components = edyn.ComponentCollection()
resolution_components.append_component(res_gauss)
resolution_model = edyn.ResolutionModel(components=resolution_components)

# The background model is created in the same way. In this case, we use
# a flat background
background_model = edyn.BackgroundModel(components=edyn.Polynomial(coefficients=[0.001]))

# We combine the resolution abd background model into an instrument
# model. This model also contains a small energy offset to account for
# instrument misalignment.

instrument_model = edyn.InstrumentModel(
    resolution_model=resolution_model,
    background_model=background_model,
)

# Collect everything into an analysis object.
vanadium_analysis = edyn.Analysis(
    display_name='Vanadium Full Analysis',
    experiment=vanadium_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

# Let us first fit a single Q index and plot the data and model to see
# how it looks
fit_result_independent_single_Q = vanadium_analysis.fit(fit_method='independent', Q_index=5)
vanadium_analysis.plot_data_and_model(Q_index=5)

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [4]:
# It looks good, so let us fit all Q indices independently and plot the
# results
fit_result_independent_all_Q = vanadium_analysis.fit(fit_method='independent')
vanadium_analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [5]:
# Inspect the Parameters as a scipp Dataset
vanadium_analysis.parameters_to_dataset()

<scipp.Dataset>
Dimensions: Sizes[Q:16, ]
Coordinates:
* Q                         float64           [1/Å]  (Q)  [0.1, 0.226667, ..., 1.87333, 2]
Data:
  DeltaFunction area        float64            [meV]  (Q)  [0.522094, 0.528523, ..., 0.537589, 0.532933]  [0.000308233, 0.000335062, ..., 0.000287459, 0.000280073]
  DeltaFunction center      float64            [meV]  (Q)  [0, 0, ..., 0, 0]  [0, 0, ..., 0, 0]
  Gaussian area             float64            [meV]  (Q)  [1, 1, ..., 1, 1]  [0, 0, ..., 0, 0]
  Gaussian center           float64            [meV]  (Q)  [0, 0, ..., 0, 0]  [0, 0, ..., 0, 0]
  Gaussian width            float64            [meV]  (Q)  [0.102228, 0.0998022, ..., 0.103462, 0.10188]  [9.25122e-06, 8.81463e-06, ..., 8.02343e-06, 7.89726e-06]
  Polynomial_c0             float64  [dimensionless]  (Q)  [0.0994206, 0.0948234, ..., 0.0976454, 0.100519]  [5.53666e-06, 5.6978e-06, ..., 4.94578e-06, 4.98191e-06]
  energy_offset             float64            [meV]  (Q)  [0.00135435, -0.000525217, ..., -0.001399, 0.000553493]  [1.33891e-05, 1.37164e-05, ..., 1.20573e-05, 1.16915e-05]
  instrument_scale          float64  [dimensionless]  (Q)  [1, 1, ..., 1, 1]  [0, 0, ..., 0, 0]

In [6]:
vanadium_analysis.get_parameters_near_bounds()

[]

In [7]:
# Plot some of fitted parameters as a function of Q
vanadium_analysis.plot_parameters(names=['DeltaFunction area'])

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [8]:
vanadium_analysis.plot_parameters(names=['Gaussian width'])

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [9]:
vanadium_analysis.plot_parameters(names=['energy_offset'])

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [10]:
# Now it's time to look at the data we want to fit. We first load the
# data
diffusion_experiment = edyn.Experiment('Diffusion')

file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/diffusion_data_example.h5',
    known_hash='5fe846b19aacbda8b8b936eb2e5310d025dc56c25b0b353521e7d6b921f229ab',
)

diffusion_experiment.load_hdf5(filename=file_path)

In [11]:
# Now we set up the model, similarly to how we set up the model for the
# vanadium data.

delta_function = edyn.DeltaFunction(display_name='DeltaFunction', area=0.2)
lorentzian = edyn.Lorentzian(display_name='Lorentzian', area=0.5, width=0.3)
component_collection = edyn.ComponentCollection(
    components=[delta_function, lorentzian],
)

sample_model = edyn.SampleModel(
    components=component_collection,
)

background_model = edyn.BackgroundModel(components=edyn.Polynomial(coefficients=[0.001]))

instrument_model = edyn.InstrumentModel(
    background_model=background_model,
)

diffusion_analysis = edyn.Analysis(
    display_name='Diffusion Full Analysis',
    experiment=diffusion_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

diffusion_analysis.instrument_model.resolution_model = (
    vanadium_analysis.instrument_model.resolution_model
)

# We fix all parameters of the resolution model.
diffusion_analysis.instrument_model.resolution_model.fix_all_parameters()

In [12]:
# Let us see how good the starting parameters are
diffusion_analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [13]:
# Now we fit the data and plot the result. Looks good!
diffusion_analysis.fit(fit_method='independent')
diffusion_analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [14]:
# Let us look at the most interesting fit parameters
diffusion_analysis.plot_parameters(names=['Lorentzian width', 'Lorentzian area'])

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [15]:
# Let us now fit directly to a diffusion model. We replace the
# Lorentzian with a Brownian translational diffusion model and keep the
# other parameters the same.
delta_function = edyn.DeltaFunction(display_name='DeltaFunction', area=0.2)
component_collection = edyn.ComponentCollection(
    components=[delta_function],
)
diffusion_model = edyn.BrownianTranslationalDiffusion(
    display_name='Brownian Translational Diffusion', diffusion_coefficient=2.4e-9, scale=0.5
)

sample_model = edyn.SampleModel(
    components=component_collection,
    diffusion_models=diffusion_model,
)

background_model = edyn.BackgroundModel(components=edyn.Polynomial(coefficients=[0.001]))

instrument_model = edyn.InstrumentModel(
    background_model=background_model,
)

diffusion_model_analysis = edyn.Analysis(
    display_name='Diffusion Full Analysis',
    experiment=diffusion_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

diffusion_model_analysis.instrument_model.resolution_model = (
    vanadium_analysis.instrument_model.resolution_model
)
diffusion_model_analysis.instrument_model.resolution_model.fix_all_parameters()

# Let us see how good the starting parameters are
diffusion_model_analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [16]:
# We now fit all the data simultaneously to the diffusion model, then
# plot the result. Looks good.
diffusion_model_analysis.fit(fit_method='simultaneous')
diffusion_model_analysis.plot_data_and_model()

WidgetFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', layout=…

In [17]:
# Let us look at the fitted diffusion coefficient
diffusion_model.get_fittable_parameters()

[<Parameter 'diffusion_coefficient': 1.126e-08 ± 9.799e-11 m^2/s, bounds=[0.0:inf]>,
 <Parameter 'scale': 0.6938 ± 0.0043 meV, bounds=[0.0:inf]>]